In [1]:
import mlflow

mlflow.set_tracking_uri("file:./mlruns")

In [2]:
# Set or create an experiment
mlflow.set_experiment("Exp 4 - Handling Imbalanced Data")

c:\Users\Soham\Documents\youtube comment analyzer\yt_env\Lib\site-packages\mlflow\tracking\_tracking_service\utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
2026/05/16 22:14:28 INFO mlflow.tracking.fluent: Experiment with name 'Exp 4 - Handling Imbalanced Data' does not exist. Creating a new experiment.


<Experiment: artifact_location=('file:c:/Users/Soham/Documents/youtube comment '
 'analyzer/mlruns/174695869595120457'), creation_time=1778949868277, experiment_id='174695869595120457', last_update_time=1778949868277, lifecycle_stage='active', name='Exp 4 - Handling Imbalanced Data', tags={}, trace_location=None, workspace='default'>

In [8]:
# =========================================================
# IMPORTS
# =========================================================
import mlflow
import mlflow.sklearn
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score
)

from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression

from imblearn.over_sampling import RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler

In [5]:
df = pd.read_csv('reddit_preprocessing.csv').dropna(subset=['clean_comment'])
df.shape

(36662, 2)

In [ ]:
# Step 1: Function to run the experiment
def run_imbalanced_experiment(imbalance_method):
    ngram_range = (1, 3)  # Trigram setting
    max_features = 10000  # Set max_features to 1000 for TF-IDF

    # Step 4: Train-test split before vectorization and resampling
    X_train, X_test, y_train, y_test = train_test_split(df['clean_comment'], df['category'], test_size=0.2, random_state=42, stratify=df['category'])

    # Step 2: Vectorization using TF-IDF, fit on training data only
    vectorizer = TfidfVectorizer(ngram_range=ngram_range, max_features=max_features)
    X_train_vec = vectorizer.fit_transform(X_train)  # Fit on training data
    X_test_vec = vectorizer.transform(X_test)  # Transform test data

    # Step 3: Handle class imbalance based on the selected method (only applied to the training set)
    if imbalance_method == 'class_weights':
        # Use class_weight in Random Forest
        class_weight = 'balanced'
    else:
        class_weight = None  # Do not apply class_weight if using resampling

        # Resampling Techniques (only apply to the training set)
        if imbalance_method == 'oversampling':
            smote = SMOTE(random_state=42)
            X_train_vec, y_train = smote.fit_resample(X_train_vec, y_train)
        elif imbalance_method == 'adasyn':
            adasyn = ADASYN(random_state=42)
            X_train_vec, y_train = adasyn.fit_resample(X_train_vec, y_train)
        elif imbalance_method == 'undersampling':
            rus = RandomUnderSampler(random_state=42)
            X_train_vec, y_train = rus.fit_resample(X_train_vec, y_train)
        elif imbalance_method == 'smote_enn':
            smote_enn = SMOTEENN(random_state=42)
            X_train_vec, y_train = smote_enn.fit_resample(X_train_vec, y_train)

    # Step 5: Define and train a Random Forest model
    with mlflow.start_run() as run:
        # Set tags for the experiment and run
        mlflow.set_tag("mlflow.runName", f"Imbalance_{imbalance_method}_RandomForest_TFIDF_Trigrams")
        mlflow.set_tag("experiment_type", "imbalance_handling")
        mlflow.set_tag("model_type", "RandomForestClassifier")

        # Add a description
        mlflow.set_tag("description", f"RandomForest with TF-IDF Trigrams, imbalance handling method={imbalance_method}")

        # Log vectorizer parameters
        mlflow.log_param("vectorizer_type", "TF-IDF")
        mlflow.log_param("ngram_range", ngram_range)
        mlflow.log_param("vectorizer_max_features", max_features)

        # Log Random Forest parameters
        n_estimators = 200
        max_depth = 15

        mlflow.log_param("n_estimators", n_estimators)
        mlflow.log_param("max_depth", max_depth)
        mlflow.log_param("imbalance_method", imbalance_method)

        # Initialize and train the model
        model = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=42, class_weight=class_weight)
        model.fit(X_train_vec, y_train)

        # Step 6: Make predictions and log metrics
        y_pred = model.predict(X_test_vec)

        # Log accuracy
        accuracy = accuracy_score(y_test, y_pred)
        mlflow.log_metric("accuracy", accuracy)

        # Log classification report
        classification_rep = classification_report(y_test, y_pred, output_dict=True)
        for label, metrics in classification_rep.items():
            if isinstance(metrics, dict):
                for metric, value in metrics.items():
                    mlflow.log_metric(f"{label}_{metric}", value)

        # Log confusion matrix
        conf_matrix = confusion_matrix(y_test, y_pred)
        plt.figure(figsize=(8, 6))
        sns.heatmap(conf_matrix, annot=True, fmt="d", cmap="Blues")
        plt.xlabel("Predicted")
        plt.ylabel("Actual")
        plt.title(f"Confusion Matrix: TF-IDF Trigrams, Imbalance={imbalance_method}")
        confusion_matrix_filename = f"confusion_matrix_{imbalance_method}.png"
        plt.savefig(confusion_matrix_filename)
        mlflow.log_artifact(confusion_matrix_filename)
        plt.close()

        # Log the model
        mlflow.sklearn.log_model(model, f"random_forest_model_tfidf_trigrams_imbalance_{imbalance_method}")

# Step 7: Run experiments for different imbalance methods
imbalance_methods = ['class_weights', 'oversampling', 'adasyn', 'undersampling', 'smote_enn']

for method in imbalance_methods:
    run_imbalanced_experiment(method)


KeyboardInterrupt: 

In [12]:
# =========================================================
# TRAIN TEST SPLIT
# =========================================================
X_train, X_test, y_train, y_test = train_test_split(
    df['clean_comment'],
    df['category'],
    test_size=0.2,
    random_state=42,
    stratify=df['category']
)

# =========================================================
# TF-IDF SETTINGS
# =========================================================
ngram_range = (1, 3)
max_features = 5000

# =========================================================
# FUNCTION TO RUN EXPERIMENT
# =========================================================
def run_experiment(model_name, imbalance_method):

    # -----------------------------------------------------
    # TF-IDF VECTORIZATION
    # -----------------------------------------------------
    vectorizer = TfidfVectorizer(
        ngram_range=ngram_range,
        max_features=max_features
    )

    X_train_vec = vectorizer.fit_transform(X_train)
    X_test_vec = vectorizer.transform(X_test)

    # -----------------------------------------------------
    # HANDLE IMBALANCE
    # -----------------------------------------------------
    class_weight = None

    y_used = y_train

    if imbalance_method == "class_weights":

        class_weight = "balanced"

    elif imbalance_method == "oversampling":

        ros = RandomOverSampler(
            random_state=42
        )

        X_train_vec, y_used = ros.fit_resample(
            X_train_vec,
            y_train
        )

    elif imbalance_method == "adasyn":

        adasyn = ADASYN(
            random_state=42
        )

        X_train_vec, y_used = adasyn.fit_resample(
            X_train_vec,
            y_train
        )

    elif imbalance_method == "undersampling":

        rus = RandomUnderSampler(
            random_state=42
        )

        X_train_vec, y_used = rus.fit_resample(
            X_train_vec,
            y_train
        )

    elif imbalance_method == "smote_enn":

        smote_enn = SMOTEENN(
            random_state=42
        )

        X_train_vec, y_used = smote_enn.fit_resample(
            X_train_vec,
            y_train
        )

    # -----------------------------------------------------
    # MODEL SELECTION
    # -----------------------------------------------------
    if model_name == "RandomForest":

        model = RandomForestClassifier(
            n_estimators=400,
            max_depth=None,
            min_samples_split=2,
            min_samples_leaf=1,
            max_features='sqrt',
            class_weight=class_weight,
            random_state=42,
            n_jobs=-1
        )

    elif model_name == "LinearSVM":

        model = LinearSVC(
            class_weight=class_weight,
            random_state=42
        )

    elif model_name == "LogisticRegression":

        model = LogisticRegression(
            class_weight=class_weight,
            max_iter=2000,
            random_state=42,
            n_jobs=-1
        )

    # -----------------------------------------------------
    # MLFLOW RUN
    # -----------------------------------------------------
    with mlflow.start_run():

        run_name = f"{model_name}_{imbalance_method}"

        mlflow.set_tag("mlflow.runName", run_name)

        # ---------------- PARAMETERS ----------------
        mlflow.log_param("model", model_name)
        mlflow.log_param("imbalance_method", imbalance_method)
        mlflow.log_param("ngram_range", ngram_range)
        mlflow.log_param("max_features", max_features)

        # ---------------- TRAIN ----------------
        model.fit(X_train_vec, y_used)

        # ---------------- PREDICT ----------------
        y_pred = model.predict(X_test_vec)

        # ---------------- METRICS ----------------
        accuracy = accuracy_score(y_test, y_pred)

        macro_f1 = f1_score(
            y_test,
            y_pred,
            average='macro'
        )

        weighted_f1 = f1_score(
            y_test,
            y_pred,
            average='weighted'
        )

        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("macro_f1", macro_f1)
        mlflow.log_metric("weighted_f1", weighted_f1)

        # ---------------- CLASSIFICATION REPORT ----------------
        report = classification_report(
            y_test,
            y_pred,
            output_dict=True
        )

        for label, metrics in report.items():

            if isinstance(metrics, dict):

                for metric_name, metric_value in metrics.items():

                    mlflow.log_metric(
                        f"{label}_{metric_name}",
                        metric_value
                    )

        # ---------------- CONFUSION MATRIX ----------------
        cm = confusion_matrix(y_test, y_pred)

        plt.figure(figsize=(8, 6))

        sns.heatmap(
            cm,
            annot=True,
            fmt='d',
            cmap='Blues'
        )

        plt.xlabel("Predicted")
        plt.ylabel("Actual")

        plt.title(
            f"{model_name} - {imbalance_method}"
        )

        filename = f"{model_name}_{imbalance_method}_cm.png"

        plt.savefig(filename)

        mlflow.log_artifact(filename)

        plt.close()

        # ---------------- SAVE MODEL ----------------
        mlflow.sklearn.log_model(
            model,
            f"{model_name}_{imbalance_method}_model"
        )

        print("=" * 60)
        print(f"MODEL: {model_name}")
        print(f"IMBALANCE METHOD: {imbalance_method}")
        print(f"Accuracy: {accuracy:.4f}")
        print(f"Macro F1: {macro_f1:.4f}")
        print(f"Weighted F1: {weighted_f1:.4f}")
        print("=" * 60)


# =========================================================
# RUN ALL EXPERIMENTS
# =========================================================
models = [
    "RandomForest",
    "LinearSVM",
    "LogisticRegression"
]

imbalance_methods = [
    "class_weights",
    "oversampling",
    "adasyn",
    "undersampling",
    "smote_enn"
]

for model_name in models:

    for imbalance_method in imbalance_methods:

        run_experiment(
            model_name,
            imbalance_method
        )

2026/05/17 01:46:23 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/17 01:46:24 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


MODEL: RandomForest
IMBALANCE METHOD: class_weights
Accuracy: 0.8088
Macro F1: 0.7780
Weighted F1: 0.7996


2026/05/17 01:47:30 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/17 01:48:12 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/05/17 01:51:29 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: C:\Users\Soham\AppData\Local\Temp\tmpe2mhobc2\model\model.pkl, flavor: sklearn). Fall back to return ['scikit-learn==1.8.0', 'cloudpickle==3.1.2']. Set logging level to DEBUG to see the full traceback. 


MODEL: RandomForest
IMBALANCE METHOD: oversampling
Accuracy: 0.8151
Macro F1: 0.7913
Weighted F1: 0.8094


2026/05/17 01:53:31 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/17 01:53:31 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


...

MODEL: LogisticRegression
IMBALANCE METHOD: adasyn
Accuracy: 0.8201
Macro F1: 0.8056
Weighted F1: 0.8211


c:\Users\Soham\Documents\youtube comment analyzer\yt_env\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
2026/05/17 01:59:47 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/17 01:59:48 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


MODEL: LogisticRegression
IMBALANCE METHOD: undersampling
Accuracy: 0.8164
Macro F1: 0.8053
Weighted F1: 0.8154


c:\Users\Soham\Documents\youtube comment analyzer\yt_env\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
2026/05/17 02:00:28 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/17 02:00:28 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


MODEL: LogisticRegression
IMBALANCE METHOD: smote_enn
Accuracy: 0.4759
Macro F1: 0.4435
Weighted F1: 0.4152
